# Step 4: The full model

Part of the model-selection steps in this folder (run in order the first time: step1 to step7). Shared setup (data,
random 80/20 train/test split by whole days, predictors, helpers) is in `model_utils.py`. The full explanation of
every step is in `docs/model_selection.md`.

**Needs:** Step 2 (`step2_terms.json`).

The full model contains every predictor left after Step 2, significant or not. Because the target is in logs, each
coefficient converts to a **% change in pickups**: `100 × (e^coef − 1)`, about `100 × coef` for small values. For the
lags, read it as: a 10% rise in last week's pickups goes with roughly a (10 × coef)% rise now.

**Reading the table** (the `statsmodels` OLS output):
- **Header:** R², adjusted R², AIC and BIC. "Covariance Type: cluster" means standard errors clustered by day.
- **One row per coefficient** (170 in total): `coef`, standard error (`std err`), `t`, p-value (`P>|t|`) and the 95%
  confidence interval (`[0.025, 0.975]`).
- **Dummy names** show the level compared with the reference: `C(hour_of_day)[T.8]` is "8 am compared with
  midnight", `C(zone, Treatment(237))[T.161]` is "zone 161 compared with zone 237". `C(hour_wknd)` is the weekend
  daily pattern and `C(zone_wknd)` the zone weekend shift.
- **The block at the bottom** (Omnibus, Jarque-Bera, skew, kurtosis, Durbin-Watson, condition number) is printed
  automatically and isn't used here: the assumptions are checked with the plots in Step 1. The "large condition
  number" note comes from the predictors being on very different scales (e.g. sunlight in W/m² vs 0/1 dummies), not
  from the collinearity handled in Step 2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import model_utils as mu

train, test = mu.load()

In [ ]:
full_terms = mu.read("step2_terms", made_by="step2_collinearity.ipynb")
res_full = mu.ols(full_terms, clustered=True)
res_full.summary()

**Results.** The full model has 14 terms and **170 parameters**: R² = 0.916, adjusted R² = 0.916, AIC = 311,874.1,
BIC = 313,702.0.

- **Rain** (`precipitation`, coef 0.021, p < 0.001): each mm of rain in the hour raises pickups by about **2.2%**.
- **Last week's and yesterday's pickups** (`log_lag168` 0.564, `log_lag24` 0.268, both p < 0.001): 10% more pickups
  at the same zone and hour last week goes with about 5.5% more now; 10% more yesterday with about 2.6% more.
- **Not significant, kept:** `holiday` (−0.169, about −15.5%, p = 0.11), `shortwave_radiation` (p = 0.06),
  `cloud_cover` (p = 0.18), `snow_depth` (p = 0.90) and `snowfall` (p = 0.92). Their confidence intervals include 0:
  the data can't rule out "no effect". The snow intervals are very wide because 2025 had few snowy hours.
- **Hour, day of week, month, zone and the weekend terms** are groups of dummies; whether each group matters is
  tested as a whole in Step 3 (all significant).